# 15 — Choice-Vigor Dissociation: Formal Tests (Phases 0-6)

Complete analysis pipeline for the choice-vigor dissociation finding.
See `instructions/quadrant_analysis_plan.md` for the full plan.

- **Phase 0:** Validation gates (reliability, robustness, identifiability)
- **Phase 1:** Characterize the 2D space (regressions, partial correlations, axes)
- **Phase 2:** Formal tests (CCA, bootstrap, β selectivity test)
- **Phase 3:** Threat modulation (cross-level LMM, per-threat correlations)
- **Phase 4:** Outcome prediction (trial-level escape, earnings)
- **Phase 5:** Affect and metacognition (confidence miscalibration)
- **Phase 6:** Psychiatric correlates (exploratory)

In [1]:
# ── SETUP ─────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from scipy.stats import spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.cross_decomposition import CCA
from statsmodels.multivariate.manova import MANOVA
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

ROOT       = Path('/Users/nokada/Desktop/EffortForagingUnderThreat')
VIGOR_PROC = ROOT / 'data' / 'exploratory_350' / 'processed' / 'vigor_processed'
VIGOR_PREP = ROOT / 'data' / 'exploratory_350' / 'processed' / 'vigor_prep'
STAGE2     = ROOT / 'data' / 'exploratory_350' / 'processed' / 'stage2_trial_processing_20260317_094210'
STAGE5     = ROOT / 'data' / 'exploratory_350' / 'processed' / 'stage5_filtered_data_20260317_094210'
STATS_DIR  = ROOT / 'results' / 'stats'
FIGS_DIR   = ROOT / 'results' / 'figs'; FIGS_DIR.mkdir(parents=True, exist_ok=True)
DPI = 150
RNG = np.random.RandomState(42)
N_BOOT = 10000

# ── Load all data ──
kp = pd.read_parquet(VIGOR_PREP / 'keypress_events.parquet')
pm = pd.read_parquet(VIGOR_PROC / 'phase_vigor_metrics.parquet')
subj_cap = pd.read_csv(VIGOR_PROC / 'subject_press_capacity.csv', index_col='subj')['capacity_pps']

with open(STAGE2 / 'processed_trials.pkl', 'rb') as f:
    pt = pickle.load(f)
pt['subj'] = pt.groupby('participantID').ngroup() + 1
pt['trial'] = pt.groupby('subj').cumcount()
enc_lookup = pt[['subj','trial','encounterTime']].rename(columns={'encounterTime':'enc_abs'})

trial_info = pm[['subj','trial','threat','isAttackTrial','outcome','choice',
                  'effort_H','distance_H','trialEndTime']].copy()

param_df = pd.DataFrame({
    p: pd.read_csv(STATS_DIR / f'FET_Exp_Bias_{p}_params.csv')
       .rename(columns={'subject':'subj'}).set_index('subj')['median']
    for p in ['z','k','beta']
})
param_df.columns = ['z','kappa','beta']

# ── Build trial-level vigor (pre-encounter, capacity-normalized, choice-adjusted) ──
kp = kp.merge(trial_info, on=['subj','trial'], how='inner')
kp = kp.merge(enc_lookup, on=['subj','trial'], how='inner')
kp = kp.merge(subj_cap.to_frame('capacity'), left_on='subj', right_index=True)

enc = kp['enc_abs']
kp_pre = kp[(kp['t'] >= enc - 2) & (kp['t'] < enc)]

tp = kp_pre.groupby(['subj','trial']).size().reset_index(name='n')
tp = tp.merge(trial_info, on=['subj','trial'])
tp = tp.merge(subj_cap.to_frame('capacity'), left_on='subj', right_index=True)
tp['rate'] = (tp['n'] / 2.0) / tp['capacity']
ch_m = tp.groupby('choice')['rate'].transform('mean')
tp['rate_adj'] = tp['rate'] / ch_m

# ── Build subject-level dataset ──
choice_subj = trial_info.groupby('subj')['choice'].mean().to_frame('p_high')
vigor_subj = tp.groupby('subj')['rate_adj'].mean().to_frame('vigor')
escape_subj = (trial_info[trial_info['isAttackTrial']==1]
               .groupby('subj')['outcome'].apply(lambda x: (x==0).mean())
               .to_frame('escape_rate'))

trial_earn = trial_info.copy()
trial_earn['reward'] = np.where(trial_earn['choice']==1, 5, 1)
trial_earn['net'] = np.where(trial_earn['outcome']==1, -5, trial_earn['reward'])
earnings = trial_earn.groupby('subj')['net'].sum().to_frame('total_earnings')

trait_affect = pd.read_csv(STATS_DIR / 'affect_trait_scores.csv').set_index('subj')

feelings = pd.read_csv(STAGE5 / 'feelings.csv')
feelings = feelings.rename(columns={'attackingProb':'p_threat','questionLabel':'affect_type','response':'rating'})
calib = {}
for subj, grp in feelings.groupby('subj'):
    for atype in ['anxiety','confidence']:
        sub = grp[grp['affect_type']==atype]
        if len(sub) >= 5:
            r, _ = spearmanr(sub['p_threat'], sub['rating'])
            calib[(subj, atype)] = r
calib_df = pd.Series(calib).unstack()
calib_df.columns = ['anx_calib','conf_calib']

psych = pd.read_csv(STAGE5 / 'psych.csv').set_index('subj')
psych_cols = ['DASS21_Anxiety','DASS21_Stress','DASS21_Depression','OASIS_Total',
              'PHQ9_Total','AMI_Total','MFIS_Physical','STICSA_Total']

subj = (choice_subj.join(vigor_subj).join(param_df).join(escape_subj)
        .join(earnings).join(trait_affect[['trait_anx','trait_conf']])
        .join(calib_df).join(psych[psych_cols]))
subj = subj.dropna(subset=['p_high','vigor','z','kappa','beta'])
subj['choice_z'] = (subj['p_high'] - subj['p_high'].mean()) / subj['p_high'].std()
subj['vigor_z'] = (subj['vigor'] - subj['vigor'].mean()) / subj['vigor'].std()
subj['cx_v'] = subj['choice_z'] * subj['vigor_z']

subj['quad'] = 'XX'
subj.loc[(subj['choice_z']>0) & (subj['vigor_z']>0), 'quad'] = 'HH'
subj.loc[(subj['choice_z']>0) & (subj['vigor_z']<=0), 'quad'] = 'HL'
subj.loc[(subj['choice_z']<=0) & (subj['vigor_z']>0), 'quad'] = 'LH'
subj.loc[(subj['choice_z']<=0) & (subj['vigor_z']<=0), 'quad'] = 'LL'

N = len(subj)
print(f'Subject-level: N={N}')
print(f'Trial-level: {len(tp):,} trials')
print(f'Quadrants: {subj["quad"].value_counts().to_dict()}')

Subject-level: N=293
Trial-level: 20,658 trials
Quadrants: {'HL': 83, 'LH': 78, 'LL': 69, 'HH': 63}


In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 0: VALIDATION GATES
# ══════════════════════════════════════════════════════════════════════════════

# Gate 0.1: Vigor reliability
vh = {}
for s, grp in tp.groupby('subj'):
    st = grp.sort_values('trial')
    vh[s] = {'odd': st.iloc[::2]['rate_adj'].mean(), 'even': st.iloc[1::2]['rate_adj'].mean()}
vh = pd.DataFrame(vh).T
r_sh, _ = stats.pearsonr(vh['odd'], vh['even'])
sb = 2*r_sh/(1+r_sh)
print(f'GATE 0.1 — Vigor split-half: r={r_sh:.3f}, SB={sb:.3f}')

# Choice reliability for comparison
ch_h = {}
for s, grp in trial_info.groupby('subj'):
    st = grp.sort_values('trial')
    ch_h[s] = {'odd': st.iloc[::2]['choice'].mean(), 'even': st.iloc[1::2]['choice'].mean()}
ch_h = pd.DataFrame(ch_h).T
r_ch, _ = stats.pearsonr(ch_h['odd'], ch_h['even'])
sb_ch = 2*r_ch/(1+r_ch)
print(f'  Choice split-half: r={r_ch:.3f}, SB={sb_ch:.3f}')
print(f'  ✅ GATE 0.1: {"PASS" if sb > 0.5 else "FAIL"} (SB={sb:.3f})')

# Gate 0.2: Independence robust to operationalization
kp_post = kp[(kp['t'] >= kp['enc_abs']) & (kp['t'] < kp['enc_abs'] + 2)]
tp_post = kp_post.groupby(['subj','trial']).size().reset_index(name='n_post')
tp_post = tp_post.merge(trial_info[['subj','trial','choice']], on=['subj','trial'])
tp_post = tp_post.merge(subj_cap.to_frame('capacity'), left_on='subj', right_index=True)
tp_post['post_rate'] = (tp_post['n_post']/2.0)/tp_post['capacity']
ch_m_post = tp_post.groupby('choice')['post_rate'].transform('mean')
tp_post['post_rate_adj'] = tp_post['post_rate'] / ch_m_post

total_pp = kp.groupby(['subj','trial']).size().reset_index(name='total_n')
tp_nocap = tp.copy()
tp_nocap['rate_nocap'] = tp_nocap['n'] / 2.0
ch_m_nc = tp_nocap.groupby('choice')['rate_nocap'].transform('mean')
tp_nocap['rate_nocap_adj'] = tp_nocap['rate_nocap'] / ch_m_nc

choice_s = trial_info.groupby('subj')['choice'].mean()
vigor_ops = {
    'Pre-enc (primary)': tp.groupby('subj')['rate_adj'].mean(),
    'Post-enc': tp_post.groupby('subj')['post_rate_adj'].mean(),
    'Total presses': total_pp.groupby('subj')['total_n'].mean(),
    'No cap norm': tp_nocap.groupby('subj')['rate_nocap_adj'].mean(),
    'No choice adj': tp.groupby('subj')['rate'].mean(),
}

n_indep = 0
print(f'\nGATE 0.2 — Independence across operationalizations:')
for label, vs in vigor_ops.items():
    m = choice_s.to_frame('c').join(vs.to_frame('v')).dropna()
    r, p = stats.pearsonr(m['c'], m['v'])
    indep = abs(r) < 0.10
    n_indep += indep
    print(f'  {label:<25s} r={r:+.3f} {"✓" if indep else "✗"}')
print(f'  ✅ GATE 0.2: {"PASS" if n_indep >= 3 else "FAIL"} ({n_indep}/5)')

# Gate 0.3: Parameter identifiability
r_kb, _ = stats.pearsonr(param_df['kappa'], param_df['beta'])
print(f'\nGATE 0.3 — k-β correlation: r={r_kb:+.3f}')
print(f'  ✅ GATE 0.3: {"PASS" if abs(r_kb) < 0.5 else "FAIL"}')

GATE 0.1 — Vigor split-half: r=0.835, SB=0.910
  Choice split-half: r=0.226, SB=0.369
  ✅ GATE 0.1: PASS (SB=0.910)

GATE 0.2 — Independence across operationalizations:
  Pre-enc (primary)         r=-0.018 ✓
  Post-enc                  r=-0.015 ✓
  Total presses             r=+0.300 ✗
  No cap norm               r=+0.040 ✓
  No choice adj             r=+0.054 ✓
  ✅ GATE 0.2: PASS (4/5)

GATE 0.3 — k-β correlation: r=+0.143
  ✅ GATE 0.3: PASS


In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 1-2: CHARACTERIZE SPACE + FORMAL TESTS
# ══════════════════════════════════════════════════════════════════════════════

X_params = subj[['z','kappa','beta']].values
X_z = (X_params - X_params.mean(0)) / X_params.std(0)

# ── 1.1: Independence with bootstrap CI ──
r_cv, p_cv = stats.pearsonr(subj['choice_z'], subj['vigor_z'])
boot_rs = [stats.pearsonr(subj['choice_z'].values[RNG.choice(N,N,replace=True)],
                            subj['vigor_z'].values[RNG.choice(N,N,replace=True)])[0]
           for _ in range(N_BOOT)]
ci = np.percentile(boot_rs, [2.5, 97.5])
print(f'1.1 Independence: r={r_cv:+.3f}, 95% CI [{ci[0]:+.3f}, {ci[1]:+.3f}], p={p_cv:.3f}')

# ── 1.2: Multiple regression asymmetry ──
for dv, label in [('choice_z','Choice'), ('vigor_z','Vigor')]:
    y = subj[dv].values
    reg = LinearRegression().fit(X_z, y)
    y_pred = reg.predict(X_z)
    ss_res = np.sum((y-y_pred)**2); ss_tot = np.sum((y-y.mean())**2)
    r2 = 1-ss_res/ss_tot; adj_r2 = 1-(1-r2)*(N-1)/(N-4)
    print(f'\n1.2 {label} ~ z+k+β: adj.R²={adj_r2:.3f}')
    for i, p_name in enumerate(['z','k','β']):
        print(f'    {p_name}: β={reg.coef_[i]:+.3f}')

# ── 1.3: Partial correlations ──
print(f'\n1.3 Partial correlations:')
print(f'  {"Param":<6s} {"→Choice":>10s} {"→Vigor":>10s}')
for pcol, plabel in [('z','z'),('kappa','k'),('beta','β')]:
    others = [c for c in ['z','kappa','beta'] if c != pcol]
    for dv in ['choice_z','vigor_z']:
        rp = LinearRegression().fit(subj[others].values, subj[pcol].values)
        rd = LinearRegression().fit(subj[others].values, subj[dv].values)
        r, _ = stats.pearsonr(subj[pcol]-rp.predict(subj[others].values),
                               subj[dv]-rd.predict(subj[others].values))
        if dv == 'choice_z': rc = r
        else: rv = r
    print(f'  {plabel:<6s} {rc:>+10.3f} {rv:>+10.3f}')

# ── 1.4: Diagonal/off-diagonal ──
subj['diag'] = subj['choice_z'] + subj['vigor_z']
subj['offdiag'] = subj['vigor_z'] - subj['choice_z']
print(f'\n1.4 Axes:')
for axis, label in [('diag','Diagonal (HH↔LL)'), ('offdiag','Off-diag (LH↔HL)')]:
    y = subj[axis].values
    reg = LinearRegression().fit(X_z, y)
    y_pred = reg.predict(X_z)
    r2 = 1-np.sum((y-y_pred)**2)/np.sum((y-y.mean())**2)
    adj_r2 = 1-(1-r2)*(N-1)/(N-4)
    print(f'  {label}: R²={adj_r2:.3f}, z={reg.coef_[0]:+.3f}, k={reg.coef_[1]:+.3f}, β={reg.coef_[2]:+.3f}')

# ── 2.1: CCA ──
print(f'\n2.1 CCA:')
cca = CCA(n_components=2).fit(X_z, subj[['choice_z','vigor_z']].values)
Xc, Yc = cca.transform(X_z, subj[['choice_z','vigor_z']].values)
for i in range(2):
    r, p = stats.pearsonr(Xc[:,i], Yc[:,i])
    print(f'  Dim {i+1}: r={r:.3f}, p={p:.2e}')
print(f'  X loadings: z={cca.x_loadings_[0,0]:+.3f}/{cca.x_loadings_[0,1]:+.3f}, '
      f'k={cca.x_loadings_[1,0]:+.3f}/{cca.x_loadings_[1,1]:+.3f}, '
      f'β={cca.x_loadings_[2,0]:+.3f}/{cca.x_loadings_[2,1]:+.3f}')
print(f'  Y loadings: choice={cca.y_loadings_[0,0]:+.3f}/{cca.y_loadings_[0,1]:+.3f}, '
      f'vigor={cca.y_loadings_[1,0]:+.3f}/{cca.y_loadings_[1,1]:+.3f}')

# ── 2.2-2.3: Bootstrap + β selectivity test ──
boot_coefs = {'choice': [], 'vigor': []}
for _ in range(N_BOOT):
    idx = RNG.choice(N, N, replace=True)
    for dv in ['choice_z','vigor_z']:
        reg = LinearRegression().fit(X_z[idx], subj[dv].values[idx])
        boot_coefs[dv.replace('_z','')].append(reg.coef_)
boot_coefs = {k: np.array(v) for k, v in boot_coefs.items()}

print(f'\n2.2 Bootstrap CIs:')
print(f'  {"Param":<4s} {"→Choice [CI]":>28s} {"→Vigor [CI]":>28s}')
for i, p_name in enumerate(['z','k','β']):
    ch = boot_coefs['choice'][:,i]; vi = boot_coefs['vigor'][:,i]
    print(f'  {p_name:<4s} {ch.mean():+.3f} [{np.percentile(ch,2.5):+.3f},{np.percentile(ch,97.5):+.3f}]  '
          f'{vi.mean():+.3f} [{np.percentile(vi,2.5):+.3f},{np.percentile(vi,97.5):+.3f}]')

print(f'\n2.3 β selectivity test:')
beta_diff = boot_coefs['choice'][:,2] - boot_coefs['vigor'][:,2]
ci_d = np.percentile(beta_diff, [2.5, 97.5])
p_diff = 2*min((beta_diff>0).mean(), (beta_diff<0).mean())
print(f'  β_choice − β_vigor = {beta_diff.mean():+.3f} [{ci_d[0]:+.3f}, {ci_d[1]:+.3f}]')
print(f'  CI excludes zero: {"YES ✅" if ci_d[0]>0 or ci_d[1]<0 else "NO"}, p={p_diff:.4f}')

1.1 Independence: r=-0.018, 95% CI [-0.115, +0.115], p=0.764

1.2 Choice ~ z+k+β: adj.R²=0.823
    z: β=-0.274
    k: β=-0.685
    β: β=-0.394

1.2 Vigor ~ z+k+β: adj.R²=0.075
    z: β=+0.199
    k: β=-0.211
    β: β=+0.142

1.3 Partial correlations:
  Param     →Choice     →Vigor
  z          -0.544     +0.202
  k          -0.849     -0.212
  β          -0.682     +0.145

1.4 Axes:
  Diagonal (HH↔LL): R²=0.481, z=-0.074, k=-0.896, β=-0.252
  Off-diag (LH↔HL): R²=0.417, z=+0.473, k=+0.473, β=+0.535

2.1 CCA:
  Dim 1: r=0.909, p=9.84e-113
  Dim 2: r=0.289, p=4.88e-07
  X loadings: z=+0.356/+0.698, k=+0.798/-0.532, β=+0.490/+0.481
  Y loadings: choice=-1.000/-0.051, vigor=-0.034/+0.999

2.2 Bootstrap CIs:
  Param                 →Choice [CI]                  →Vigor [CI]
  z    -0.276 [-0.321,-0.238]  +0.201 [+0.097,+0.311]
  k    -0.687 [-0.729,-0.646]  -0.211 [-0.329,-0.097]
  β    -0.408 [-0.571,-0.310]  +0.145 [+0.019,+0.270]

2.3 β selectivity test:
  β_choice − β_vigor = -0.553 [-0.

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 3: THREAT MODULATION
# ══════════════════════════════════════════════════════════════════════════════

# Add subject-level choice to trial data
tp['choice_subj'] = tp['subj'].map(trial_info.groupby('subj')['choice'].mean())
tp['choice_subj_z'] = (tp['choice_subj'] - tp['choice_subj'].mean()) / tp['choice_subj'].std()
tp['threat_z'] = (tp['threat'] - tp['threat'].mean()) / tp['threat'].std()
tp['dist_z'] = (tp['distance_H'] - tp['distance_H'].mean()) / tp['distance_H'].std()
tp['vigor_trial_z'] = (tp['rate_adj'] - tp['rate_adj'].mean()) / tp['rate_adj'].std()

# 3.1: Cross-level LMM
print('3.1 Cross-level LMM:')
model = smf.mixedlm('vigor_trial_z ~ choice_subj_z * threat_z + dist_z',
                      tp, groups=tp['subj'])
fit = model.fit(reml=False)
for term in ['choice_subj_z','threat_z','dist_z','choice_subj_z:threat_z']:
    b, p = fit.params[term], fit.pvalues[term]
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else ''
    print(f'  {term:<30s} β={b:+.4f}, p={p:.2e} {sig}')

# 3.2: Per-threat correlations
print(f'\n3.2 Per-threat correlations:')
for th in [0.1, 0.5, 0.9]:
    ch_th = trial_info[trial_info['threat']==th].groupby('subj')['choice'].mean()
    vig_th = tp[tp['threat']==th].groupby('subj')['rate_adj'].mean()
    m = ch_th.to_frame('c').join(vig_th.to_frame('v')).dropna()
    r, p = stats.pearsonr(m['c'], m['v'])
    
    boot = [stats.pearsonr(m.values[RNG.choice(len(m),len(m),replace=True),0],
                            m.values[RNG.choice(len(m),len(m),replace=True),1])[0]
            for _ in range(N_BOOT)]
    ci = np.percentile(boot, [2.5, 97.5])
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else ''
    print(f'  T={th}: r={r:+.3f} [{ci[0]:+.3f},{ci[1]:+.3f}] p={p:.3f} {sig}')

# Fisher z-test
r_lo = stats.pearsonr(*[trial_info[trial_info['threat']==0.1].groupby('subj')['choice'].mean().to_frame('c')
                          .join(tp[tp['threat']==0.1].groupby('subj')['rate_adj'].mean().to_frame('v')).dropna()[c]
                         for c in ['c','v']])[0]
r_hi = stats.pearsonr(*[trial_info[trial_info['threat']==0.9].groupby('subj')['choice'].mean().to_frame('c')
                          .join(tp[tp['threat']==0.9].groupby('subj')['rate_adj'].mean().to_frame('v')).dropna()[c]
                         for c in ['c','v']])[0]
z_fish = (np.arctanh(r_lo) - np.arctanh(r_hi)) / np.sqrt(2/(N-3))
p_fish = 2*(1-stats.norm.cdf(abs(z_fish)))
print(f'\n  Fisher z-test (low vs high): z={z_fish:.2f}, p={p_fish:.4f}')

# 3.3: β mediates reversal
print(f'\n3.3 β→choice and β→vigor by threat:')
for th in [0.1, 0.5, 0.9]:
    ch_th = trial_info[trial_info['threat']==th].groupby('subj')['choice'].mean()
    vig_th = tp[tp['threat']==th].groupby('subj')['rate_adj'].mean()
    m = ch_th.to_frame('c').join(vig_th.to_frame('v')).join(param_df).dropna()
    r_bc, _ = stats.pearsonr(m['beta'], m['c'])
    r_bv, _ = stats.pearsonr(m['beta'], m['v'])
    print(f'  T={th}: β→choice={r_bc:+.3f}, β→vigor={r_bv:+.3f}')

3.1 Cross-level LMM:
  choice_subj_z                  β=-0.0088, p=7.70e-01 
  threat_z                       β=-0.0284, p=2.97e-06 ***
  dist_z                         β=-0.0373, p=8.54e-10 ***
  choice_subj_z:threat_z         β=-0.0215, p=3.98e-04 ***

3.2 Per-threat correlations:
  T=0.1: r=+0.196 [-0.114,+0.111] p=0.001 ***
  T=0.5: r=+0.013 [-0.113,+0.114] p=0.821 
  T=0.9: r=-0.219 [-0.114,+0.114] p=0.000 ***

  Fisher z-test (low vs high): z=5.07, p=0.0000

3.3 β→choice and β→vigor by threat:
  T=0.1: β→choice=-0.214, β→vigor=+0.100
  T=0.5: β→choice=-0.522, β→vigor=+0.084
  T=0.9: β→choice=-0.505, β→vigor=+0.130


In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 4: OUTCOME PREDICTION
# ══════════════════════════════════════════════════════════════════════════════

# 4.1: Trial-level escape
tp_atk = tp[tp['isAttackTrial']==1].copy()
tp_atk['escaped'] = (tp_atk['outcome']==0).astype(float)
tp_atk['choice_z_trial'] = (tp_atk['choice'].astype(float) - tp_atk['choice'].mean()) / tp_atk['choice'].std()

print('4.1 Trial-level escape LMM:')
fit = smf.mixedlm('escaped ~ vigor_trial_z + choice_z_trial + threat_z + dist_z',
                    tp_atk, groups=tp_atk['subj']).fit(reml=False)
for term in ['vigor_trial_z','choice_z_trial','threat_z','dist_z']:
    b, p = fit.params[term], fit.pvalues[term]
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else ''
    print(f'  {term:<20s} β={b:+.4f}, p={p:.2e} {sig}')

# Model comparison
fit_v = smf.mixedlm('escaped ~ vigor_trial_z + threat_z + dist_z', tp_atk, groups=tp_atk['subj']).fit(reml=False)
fit_c = smf.mixedlm('escaped ~ choice_z_trial + threat_z + dist_z', tp_atk, groups=tp_atk['subj']).fit(reml=False)
print(f'\n  AIC: vigor-only={fit_v.aic:.0f}, choice-only={fit_c.aic:.0f}, both={fit.aic:.0f}')
print(f'  Adding vigor to choice: ΔAIC={fit_c.aic - fit.aic:.0f}')

# 4.2: Subject-level earnings
print(f'\n4.2 Earnings:')
X_e = subj[['choice_z','vigor_z','cx_v']].values
y_e = (subj['total_earnings'].values - subj['total_earnings'].mean()) / subj['total_earnings'].std()
reg_e = LinearRegression().fit(X_e, y_e)
r2_e = 1 - np.sum((y_e - reg_e.predict(X_e))**2) / np.sum((y_e - y_e.mean())**2)
print(f'  R²={r2_e:.3f}, choice β={reg_e.coef_[0]:+.3f}, vigor β={reg_e.coef_[1]:+.3f}, C×V β={reg_e.coef_[2]:+.3f}')

# 4.3: Pairwise
print(f'\n4.3 Pairwise (same choice, different vigor):')
for q1, q2 in [('HH','HL'),('LH','LL')]:
    s1 = subj[subj['quad']==q1]['escape_rate']; s2 = subj[subj['quad']==q2]['escape_rate']
    t, p = stats.ttest_ind(s1, s2)
    print(f'  {q1} vs {q2}: {s1.mean():.0%} vs {s2.mean():.0%}, t={t:.1f}, p={p:.1e}')

print(f'\nPairwise (same vigor, different choice):')
for q1, q2 in [('HH','LH'),('HL','LL')]:
    s1 = subj[subj['quad']==q1]['escape_rate']; s2 = subj[subj['quad']==q2]['escape_rate']
    t, p = stats.ttest_ind(s1, s2)
    print(f'  {q1} vs {q2}: {s1.mean():.0%} vs {s2.mean():.0%}, t={t:.1f}, p={p:.2f}')

4.1 Trial-level escape LMM:
  vigor_trial_z        β=+0.0923, p=1.19e-77 ***
  choice_z_trial       β=-0.1768, p=0.00e+00 ***
  threat_z             β=+0.0177, p=5.43e-04 ***
  dist_z               β=-0.0203, p=1.67e-06 ***

  AIC: vigor-only=12808, choice-only=11627, both=11286
  Adding vigor to choice: ΔAIC=341

4.2 Earnings:
  R²=0.604, choice β=+0.208, vigor β=+0.757, C×V β=+0.036

4.3 Pairwise (same choice, different vigor):
  HH vs HL: 53% vs 19%, t=12.7, p=2.9e-25
  LH vs LL: 60% vs 25%, t=12.6, p=3.7e-25

Pairwise (same vigor, different choice):
  HH vs LH: 53% vs 60%, t=-2.3, p=0.02
  HL vs LL: 19% vs 25%, t=-2.3, p=0.02


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 5: AFFECT AND METACOGNITION
# ══════════════════════════════════════════════════════════════════════════════

# 5.1: Confidence miscalibration
subj['conf_z'] = (subj['trait_conf'] - subj['trait_conf'].mean()) / subj['trait_conf'].std()
subj['escape_z'] = (subj['escape_rate'] - subj['escape_rate'].mean()) / subj['escape_rate'].std()
subj['conf_bias'] = subj['conf_z'] - subj['escape_z']

print('5.1 Confidence miscalibration:')
for q in ['HH','HL','LH','LL']:
    s = subj[subj['quad']==q]
    print(f'  {q}: bias={s["conf_bias"].mean():+.3f}')

groups = [subj[subj['quad']==q]['conf_bias'].dropna().values for q in ['HH','HL','LH','LL']]
F, p = stats.f_oneway(*groups)
print(f'  ANOVA: F={F:.1f}, p={p:.1e}')

cb_clean = subj[['choice_z','vigor_z','cx_v','conf_bias']].dropna()
reg_cb = LinearRegression().fit(cb_clean[['choice_z','vigor_z','cx_v']].values, cb_clean['conf_bias'].values)
r2_cb = 1 - np.sum((cb_clean['conf_bias'] - reg_cb.predict(cb_clean[['choice_z','vigor_z','cx_v']].values))**2) / \
        np.sum((cb_clean['conf_bias'] - cb_clean['conf_bias'].mean())**2)
print(f'  R²={r2_cb:.3f}: choice β={reg_cb.coef_[0]:+.3f}, vigor β={reg_cb.coef_[1]:+.3f}, C×V β={reg_cb.coef_[2]:+.3f}')

# 5.2: Anxiety calibration
print(f'\n5.2 Anxiety calibration:')
ac_clean = subj[['choice_z','vigor_z','cx_v','anx_calib']].dropna()
reg_ac = LinearRegression().fit(ac_clean[['choice_z','vigor_z','cx_v']].values, ac_clean['anx_calib'].values)
r2_ac = 1 - np.sum((ac_clean['anx_calib'] - reg_ac.predict(ac_clean[['choice_z','vigor_z','cx_v']].values))**2) / \
        np.sum((ac_clean['anx_calib'] - ac_clean['anx_calib'].mean())**2)
print(f'  R²={r2_ac:.3f}: choice β={reg_ac.coef_[0]:+.3f}, vigor β={reg_ac.coef_[1]:+.3f}')
for q in ['HH','HL','LH','LL']:
    print(f'  {q}: anx_calib={subj[subj["quad"]==q]["anx_calib"].mean():+.3f}')

# 5.3: Affect in choice-vigor space
print(f'\n5.3 Affect:')
for dv, label in [('trait_conf','Confidence'),('trait_anx','Anxiety')]:
    clean = subj[['choice_z','vigor_z','cx_v',dv]].dropna()
    y = (clean[dv] - clean[dv].mean()) / clean[dv].std()
    reg = LinearRegression().fit(clean[['choice_z','vigor_z','cx_v']].values, y.values)
    r2 = 1 - np.sum((y - reg.predict(clean[['choice_z','vigor_z','cx_v']].values))**2) / np.sum((y - y.mean())**2)
    print(f'  {label}: R²={r2:.3f}, choice={reg.coef_[0]:+.3f}, vigor={reg.coef_[1]:+.3f}, C×V={reg.coef_[2]:+.3f}')

5.1 Confidence miscalibration:
  HH: bias=-0.247
  HL: bias=+0.981
  LH: bias=-1.177
  LL: bias=+0.377
  ANOVA: F=50.2, p=3.8e-26
  R²=0.415: choice β=+0.423, vigor β=-0.783, C×V β=+0.130

5.2 Anxiety calibration:
  R²=0.054: choice β=+0.026, vigor β=+0.078
  HH: anx_calib=+0.425
  HL: anx_calib=+0.224
  LH: anx_calib=+0.362
  LL: anx_calib=+0.275

5.3 Affect:
  Confidence: R²=0.077, choice=+0.264, vigor=+0.011, C×V=+0.113
  Anxiety: R²=0.047, choice=-0.108, vigor=-0.173, C×V=-0.112


In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 6: PSYCHIATRIC CORRELATES (exploratory)
# ══════════════════════════════════════════════════════════════════════════════

print('6. Psychiatric measures ~ choice + vigor + interaction:')
print(f'  {"Measure":<22s} {"Choice":>8s} {"Vigor":>8s} {"C×V":>8s} {"R²":>6s} {"p":>8s}')
print('  ' + '-'*62)

p_vals = []
for pc_col in psych_cols:
    clean = subj[['choice_z','vigor_z','cx_v',pc_col]].dropna()
    if len(clean) < 50: continue
    y = (clean[pc_col] - clean[pc_col].mean()) / clean[pc_col].std()
    X = clean[['choice_z','vigor_z','cx_v']].values
    reg = LinearRegression().fit(X, y)
    y_pred = reg.predict(X)
    n = len(y); ss_res = np.sum((y-y_pred)**2); ss_tot = np.sum((y-y.mean())**2)
    r2 = 1-ss_res/ss_tot; adj_r2 = 1-(1-r2)*(n-1)/(n-4)
    F = (ss_tot-ss_res)/3 / (ss_res/(n-4))
    p_F = 1 - stats.f.cdf(F, 3, n-4)
    p_vals.append(p_F)
    sig = '**' if p_F<0.01 else '*' if p_F<0.05 else ''
    print(f'  {pc_col:<22s} {reg.coef_[0]:>+7.3f}  {reg.coef_[1]:>+7.3f}  {reg.coef_[2]:>+7.3f}  {adj_r2:>5.3f}  {p_F:>7.3f} {sig}')

reject, p_fdr, _, _ = multipletests(p_vals, method='fdr_bh')
print(f'\n  FDR-corrected: {sum(reject)}/{len(reject)} significant')

# ══════════════════════════════════════════════════════════════════════════════
# SUMMARY TABLE
# ══════════════════════════════════════════════════════════════════════════════
print(f'\n{"═"*70}')
print('FULL SUMMARY')
print('═'*70)
print(f"""
  Phase 0: All gates passed (vigor SB=0.91, 4/5 independent, k-β r=0.14)
  
  Phase 1-2:
    Independence: r={subj['choice_z'].corr(subj['vigor_z']):+.3f}
    Params→choice: R²=0.82 (k dominant)
    Params→vigor:  R²=0.08 (11x weaker)
    β selectivity: diff=−0.56 [−0.80,−0.38], p<0.0001
    CCA: 2 dimensions (r=0.91, r=0.29)
    
  Phase 3: Threat reversal
    LMM interaction: β=−0.022, p=0.0004
    r swings from +0.20 to −0.22 (Fisher z=5.07, p<0.0001)
    β→choice amplifies with threat (−0.21→−0.52), β→vigor flat
    
  Phase 4: Outcomes
    Trial escape: vigor β=+0.09 (p=10⁻⁷⁷), choice β=−0.18
    Vigor triples escape within same choice (p=10⁻²⁵)
    Earnings: R²=0.60, vigor β=+0.76
    
  Phase 5: Affect
    Confidence miscalibration: R²=0.42 (F=50, p=10⁻²⁶)
    HL overconfident (+0.98), LH underconfident (−1.18)
    
  Phase 6: Psychiatric
    AMI ~ vigor (β=+0.31), survives FDR
""")

6. Psychiatric measures ~ choice + vigor + interaction:
  Measure                  Choice    Vigor      C×V     R²        p
  --------------------------------------------------------------
  DASS21_Anxiety          +0.048   -0.053   -0.068  -0.001    0.428 
  DASS21_Stress           +0.047   +0.032   -0.064  -0.002    0.491 
  DASS21_Depression       +0.034   +0.120   -0.047  0.009    0.130 
  OASIS_Total             +0.044   -0.001   -0.047  -0.006    0.731 
  PHQ9_Total              +0.033   +0.104   -0.078  0.010    0.117 
  AMI_Total               -0.056   +0.311   -0.024  0.093    0.000 **
  MFIS_Physical           +0.029   -0.020   -0.018  -0.009    0.927 
  STICSA_Total            +0.015   -0.062   -0.075  -0.001    0.461 

  FDR-corrected: 1/8 significant

══════════════════════════════════════════════════════════════════════
FULL SUMMARY
══════════════════════════════════════════════════════════════════════

  Phase 0: All gates passed (vigor SB=0.91, 4/5 independent, k-β r=0.